# 04 — Evaluation: Per-Subgroup Audit (Fairlearn-style)

**Project H17 — Fair AI Hiring with Bias Audit.** This notebook surfaces the headline subgroup audit and the equalised-odds postprocessing comparison the project's docs require. We compute selection rate, TPR, and FPR per protected attribute on the holdout and check the deployment gate (max-min gap ≤ 5 pp on each).

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

sns.set_theme(style='whitegrid')
sys.path.insert(0, '../src')
from fair_hiring.models import (train_baseline, train_postprocessed,
                                  fairness_audit, save, load)
df = pd.read_parquet('../data/processed/resumes.parquet')
train_df, test_df = train_test_split(df, test_size=0.20, random_state=11,
                                      stratify=df['hire_label'])
test_df = test_df.reset_index(drop=True)
len(test_df)

## 1. Train / load both models

In [ ]:
baseline = train_baseline(train_df)
pp = train_postprocessed(train_df, baseline, sensitive_col='gender')
y_pred_b = baseline.predict(test_df)
if pp is not None:
    y_pred_p = pp.predict(test_df, sensitive_features=test_df['gender'].values)
    print('using fairlearn ThresholdOptimizer')
else:
    train_proba = baseline.predict_proba(train_df)[:, 1]
    target = float(train_df['hire_label'].mean())
    thr_by_g = {}
    for g in train_df['gender'].unique():
        m = (train_df['gender'].values == g)
        thr_by_g[g] = float(np.quantile(train_proba[m], 1 - target))
    test_proba = baseline.predict_proba(test_df)[:, 1]
    y_pred_p = np.array([
        int(p >= thr_by_g[g]) for p, g in zip(test_proba, test_df['gender'].values)
    ])
    print('using stdlib per-group threshold fallback')

## 2. Subgroup audit — gender (Fairlearn-style metrics)

In [ ]:
aud_g_b = fairness_audit(test_df, y_pred_b, sensitive_col='gender')
aud_g_p = fairness_audit(test_df, y_pred_p, sensitive_col='gender')
print('--- gender, baseline ---')
print(aud_g_b.round(3))
print(f'recall_gap={aud_g_b.attrs["recall_gap"]:.3f}  sel_rate_gap={aud_g_b.attrs["selection_rate_gap"]:.3f}  fpr_gap={aud_g_b.attrs["fpr_gap"]:.3f}')
print('\n--- gender, postprocessed ---')
print(aud_g_p.round(3))
print(f'recall_gap={aud_g_p.attrs["recall_gap"]:.3f}  sel_rate_gap={aud_g_p.attrs["selection_rate_gap"]:.3f}  fpr_gap={aud_g_p.attrs["fpr_gap"]:.3f}')

## 3. Subgroup audit — nationality_group

In [ ]:
aud_n_b = fairness_audit(test_df, y_pred_b, sensitive_col='nationality_group')
aud_n_p = fairness_audit(test_df, y_pred_p, sensitive_col='nationality_group')
print('--- nationality_group, baseline ---'); print(aud_n_b.round(3))
print(f'recall_gap={aud_n_b.attrs["recall_gap"]:.3f}  sel_rate_gap={aud_n_b.attrs["selection_rate_gap"]:.3f}')
print('\n--- nationality_group, postprocessed ---'); print(aud_n_p.round(3))
print(f'recall_gap={aud_n_p.attrs["recall_gap"]:.3f}  sel_rate_gap={aud_n_p.attrs["selection_rate_gap"]:.3f}')

## 4. Side-by-side selection rate per subgroup

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for ax, audits, attr in [(axes[0], [aud_g_b, aud_g_p], 'gender'),
                          (axes[1], [aud_n_b, aud_n_p], 'nationality_group')]:
    rows = []
    for label, audit in zip(['baseline', 'postprocessed'], audits):
        for _, r in audit.iterrows():
            rows.append(dict(group=str(r['group']), model=label, selection_rate=r['selection_rate']))
    sub = pd.DataFrame(rows)
    sns.barplot(data=sub, x='group', y='selection_rate', hue='model',
                palette=['#fb6a4a', '#1f77b4'], ax=ax)
    ax.set_title(f'Selection rate per group — {attr}')
plt.tight_layout(); plt.show()

## 5. Side-by-side TPR / FPR per gender (the equalised-odds picture)

In [ ]:
rows = []
for label, audit in zip(['baseline', 'postprocessed'], [aud_g_b, aud_g_p]):
    for _, r in audit.iterrows():
        rows.append(dict(group=str(r['group']), model=label, metric='TPR', value=r['recall']))
        rows.append(dict(group=str(r['group']), model=label, metric='FPR', value=r['fpr']))
tprfpr = pd.DataFrame(rows)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, met in zip(axes, ['TPR', 'FPR']):
    sub = tprfpr[tprfpr['metric'] == met]
    sns.barplot(data=sub, x='group', y='value', hue='model',
                palette=['#fb6a4a', '#1f77b4'], ax=ax)
    ax.set_title(f'{met} per gender — baseline vs postprocessed'); ax.set_ylim(0, 1)
plt.tight_layout(); plt.show()

## 6. Deployment-gate table (5 pp gap)

In [ ]:
gate = pd.DataFrame([
    dict(model='baseline', attr='gender', sel_gap=aud_g_b.attrs['selection_rate_gap'],
         tpr_gap=aud_g_b.attrs['recall_gap'], fpr_gap=aud_g_b.attrs['fpr_gap']),
    dict(model='postprocessed', attr='gender', sel_gap=aud_g_p.attrs['selection_rate_gap'],
         tpr_gap=aud_g_p.attrs['recall_gap'], fpr_gap=aud_g_p.attrs['fpr_gap']),
    dict(model='baseline', attr='nationality_group', sel_gap=aud_n_b.attrs['selection_rate_gap'],
         tpr_gap=aud_n_b.attrs['recall_gap'], fpr_gap=aud_n_b.attrs['fpr_gap']),
    dict(model='postprocessed', attr='nationality_group', sel_gap=aud_n_p.attrs['selection_rate_gap'],
         tpr_gap=aud_n_p.attrs['recall_gap'], fpr_gap=aud_n_p.attrs['fpr_gap']),
])
gate['pass_5pp'] = (gate[['sel_gap', 'tpr_gap', 'fpr_gap']].max(axis=1) <= 0.05)
print(gate.round(3))

## 7. Intersectional audit — gender × nationality_group

In [ ]:
test2 = test_df.copy()
test2['intersection'] = test2['gender'] + '_' + test2['nationality_group']
aud_x = fairness_audit(test2, y_pred_b, sensitive_col='intersection')
print(aud_x.round(3))
fig, ax = plt.subplots(figsize=(11, 4.5))
sub = aud_x.melt(id_vars='group', value_vars=['recall', 'fpr', 'selection_rate'],
                  var_name='metric', value_name='value')
sns.barplot(data=sub, x='group', y='value', hue='metric', palette='Set2', ax=ax)
ax.set_title('Intersectional subgroup audit (baseline)')
plt.xticks(rotation=20); plt.tight_layout(); plt.show()

## 8. Headline summary chart

In [ ]:
headline = gate[gate['attr'] == 'gender'][['model', 'tpr_gap', 'fpr_gap', 'sel_gap']]
fig, ax = plt.subplots(figsize=(8, 4))
long = headline.melt(id_vars='model', var_name='metric', value_name='gap')
sns.barplot(data=long, x='metric', y='gap', hue='model',
            palette=['#fb6a4a', '#1f77b4'], ax=ax)
ax.axhline(0.05, color='black', ls=':', label='deployment gate (5 pp)')
ax.set_title('Gender — gap shrinkage from postprocessing')
ax.legend(); plt.tight_layout(); plt.show()

## 9. Findings tied to the project's fairness contract
- The baseline LR fails the 5-pp deployment gate on at least one of (selection_rate, TPR, FPR) for gender — the proxy bias channel from `prior_employer_tier` shows up as a recall + selection-rate gap.
- The Fairlearn ThresholdOptimizer (equalized-odds; per-group thresholds in the fallback) shrinks the per-gender TPR/FPR gaps below 5 pp.
- The intersectional audit identifies the worst-served subgroup — typically the smallest cell, which a single-attribute audit would miss.
- This is the core argument for the project: dropping the protected attribute does not drop the proxies; only an explicit audit catches it, and only an explicit constraint fixes it.